# JARVISEO 검출기 파인튜닝 — 시드 반복 (#167)

**쓰는 법**: 런타임 → 런타임 유형 변경 → **A100 GPU**(없으면 **L4 GPU**) → 저장 → 런타임 → **모두 실행**.
중간에 Google Drive 연결 허락 창이 한 번 뜬다. 그 밖에 고를 것은 없다. 다 돌 때까지 탭을 열어 둔다.

- 왜: R4 에서 데이터 · 설정이 같고 데이터 준비 워커 수만 다른 두 학습이 가리킴 valid 62장 중 9장에서 갈렸다. R2 와 R3 의 차이(5장)도 이 폭 안이라, 같은 설정을 **시드만 바꿔** 여러 번 학습해 학습 우연의 폭부터 잰다(`docs/experiments.md` "시드 반복")
- 입력: 내 드라이브 `JARVISEO/JARVISEO_R2_data.zip` · `JARVISEO_R3_data.zip` — R2 · R3 때 맥에서 `tools/auto_label.py import` · `build` 로 만든 학습 묶음을 **그대로** 쓴다
- 학습: 묶음 r2 · r3 × 시드 0 · 1 · 2 = 6번(`r2_s0` → `r3_s0` → `r2_s1` → `r3_s1` → `r2_s2` → `r3_s2`). 시드 말고는 R2 · R3 와 같은 설정이다
- **A100 · L4 가 아니거나 CPU 가 8개 미만이면 셀 1 에서 멈춘다.** CPU 수가 데이터 준비 워커 수(최대 8)라 바뀌면 같은 seed 라도 다른 학습이 된다. R1~R4 는 T4 였지만 11/2 밤 Colab 의 T4 는 '고용량 RAM' 이 떠 있어도 CPU 2개라, 워커 8 이 나오는 고성능 GPU 로 옮겼다. 그래서 원래 R2 · R3 와는 GPU 가 달라 시드 0 재현은 보지 않고, 6개를 한 실행에서 서로 견준다
- 결과(Drive `JARVISEO/`): `detector/runs/r2_s0` ~ `r3_s2` · **`SEEDS_report.txt`(이 실행의 기록 전부)** · **`JARVISEO_seeds_weights.zip`(학습마다 best · last · results.csv — 맥으로 받을 파일)**
- 결과는 덮어쓰지 않는다. 이미 있으면 셀 2 에서 멈춘다.
- 레포 · CLIP 은 필요 없다(ultralytics 만 설치한다). 다른 묶음 · 시드로 돌릴 때는 셀 1 의 `ROUND` · `DATAS` · `SEEDS` 만 바꾼다.

In [ ]:
# 1. 설정 · 도움 함수 · GPU · CPU 확인
import csv
import os
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import UTC, datetime
from importlib import metadata
from pathlib import Path

ROUND = "seeds"
# R2 · R3 의 학습 묶음 · 설정을 그대로 두고 시드만 바꿔 여러 번 학습한다.
# 학습끼리의 차이가 곧 학습 우연의 폭이다(R4 는 워커 수만 달라도 valid 9장이 갈렸다).
DATAS = ["r2", "r3"]  # 쓸 학습 묶음(zip · yolo_<묶음> 폴더)
SEEDS = [0, 1, 2]
# 순서: r2_s0 → r3_s0 → r2_s1 → r3_s1 → r2_s2 → r3_s2. 중간에 끊겨도 두 묶음이 같이 남는다.
RUNS = {f"{data}_s{seed}": (data, seed) for seed in SEEDS for data in DATAS}
DRIVE = Path("/content/drive/MyDrive/JARVISEO")
ZIPS = {data: DRIVE / f"JARVISEO_{data.upper()}_data.zip" for data in DATAS}
WORK = Path("/content/work")  # 압축은 Colab 로컬 디스크에 푼다(Drive 보다 읽기가 빠르다)
REPORT = DRIVE / f"{ROUND.upper()}_report.txt"
OUT_ZIP = DRIVE / f"JARVISEO_{ROUND}_weights.zip"  # 맥으로 받을 파일 하나
ULTRALYTICS = "8.4.155"  # 맥에서 R0 · R1 을 잰 버전과 같게 고정한다
EPOCHS = 100
# CPU 수 = 데이터 준비 워커 수(min(CPU, 8))라 바뀌면 같은 seed 라도 다른 학습이 된다.
# R1~R4 의 T4 는 11/2 밤 '고용량 RAM' 이 떠 있어도 CPU 2개였다 → 워커 8 이 나오는
# 고성능 GPU 에서 6개를 한 번에 돌려 서로 견준다(원래 R2 · R3 와는 GPU 가 달라 견주지 않는다).
GPUS = ("A100", "L4")
MIN_CPUS = 8


def note(text=""):
    """화면에 찍고 보고서 파일에도 남긴다."""
    print(text)
    with REPORT.open("a", encoding="utf-8") as f:
        f.write(text + "\n")


if shutil.which("nvidia-smi") is None:
    raise RuntimeError("GPU 가 없습니다. 런타임 유형을 A100 GPU 로 바꾸고 다시 모두 실행하세요.")
GPU = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
).stdout.strip()
CPUS = os.cpu_count() or 0
print("GPU:", GPU, "· CPU:", CPUS)
if not any(name in GPU for name in GPUS) or CPUS < MIN_CPUS:
    raise RuntimeError(
        f"GPU {GPU} · CPU {CPUS}개입니다(필요: A100 또는 L4 · CPU {MIN_CPUS}개 이상). "
        "런타임 → 런타임 유형 변경 → A100 GPU(없으면 L4 GPU) → 저장 → 다시 모두 실행하세요."
    )

In [ ]:
# 2. Drive 연결 · 덮어쓰기 막기 (허락 창이 뜨면 내 계정으로 허용)
from google.colab import drive

drive.mount("/content/drive")
missing = [path.name for path in ZIPS.values() if not path.exists()]
if missing:
    raise RuntimeError(
        f"학습 묶음이 없습니다: {', '.join(missing)} — 맥의 data/datasets/detector/ 에 있는 "
        "zip 을 내 드라이브 JARVISEO 에 올립니다."
    )
# 결과가 이미 있으면 멈춘다. 덮어쓰면 그 학습을 같은 조건으로 다시 비교할 수 없다.
# 앞 실행이 끊겨 폴더만 남았으면 Drive 에서 그 폴더 이름을 바꿔(r2_s1_failed1 등) 두고
# 다시 돌린다.
done = [DRIVE / "detector" / "runs" / name for name in RUNS] + [OUT_ZIP]
if any(p.exists() for p in done):
    raise RuntimeError(
        "이미 결과가 있습니다(덮어쓰지 않음): " + ", ".join(str(p) for p in done if p.exists())
    )
if REPORT.exists():  # 앞 실행이 결과를 남기기 전에 멈췄다 → 그 기록은 이름을 바꿔 둔다
    REPORT.rename(REPORT.with_name(f"{REPORT.stem}_failed_{int(time.time())}.txt"))

In [ ]:
# 3. 설치 · 압축 풀기 (1~2분)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}"], check=True
)
for data, path in ZIPS.items():
    # 묶음마다 다른 폴더에 푼다 — 두 zip 모두 맨 위에 prep_report.txt 가 있다.
    # data.yaml 은 자기 폴더 기준 상대 경로(train: images/train)라 어디에 풀어도 된다.
    folder = WORK / data
    if not folder.exists():
        with zipfile.ZipFile(path) as z:
            z.extractall(folder)
    if not (folder / f"yolo_{data}" / "data.yaml").exists():
        raise RuntimeError(f"{path.name} 안에 yolo_{data}/data.yaml 이 없습니다")

note(f"# JARVISEO 검출기 {ROUND.upper()} 기록")
note(f"- 시각(UTC): {datetime.now(UTC):%Y-%m-%d %H:%M}")
note(
    f"- ultralytics {ULTRALYTICS} · torch {metadata.version('torch')} · GPU {GPU} · "
    f"CPU {CPUS}개(데이터 준비 워커 {min(CPUS, 8)})"
)
note(
    f"- 학습: yolov8n.pt 에서 · 묶음 {' · '.join(DATAS)} × 시드 {' · '.join(map(str, SEEDS))} · "
    f"imgsz 640 · {EPOCHS} epoch · patience 0 · deterministic"
)
for data in DATAS:
    note()
    note(f"## 학습 묶음 {data} (맥에서 만든 기록)")
    note((WORK / data / "prep_report.txt").read_text(encoding="utf-8"))

In [ ]:
# 4. 학습 6번 (약 40분). 결과는 Drive detector/runs/<이름> 에 바로 저장된다(끊겨도 남음)
from ultralytics import YOLO

train_minutes = {}
for name, (data, seed) in RUNS.items():
    started = time.time()
    # 지금 비서가 쓰는 가중치와 같은 구조에서 시작한다. 시드 말고는 R2 · R3 와 같은 설정이라
    # 학습끼리의 차이가 학습 우연의 폭이 된다.
    YOLO("yolov8n.pt").train(
        data=str(WORK / data / f"yolo_{data}" / "data.yaml"),
        epochs=EPOCHS,
        # 0 = 조기 종료 없음(ultralytics/utils/torch_utils.py:1112). 학습이 몇 분이라 아낄 게 없고,
        # 끝까지 돌아야 학습률이 다 내려가고 마지막 10 epoch 의 모자이크 끄기(close_mosaic)도 탄다.
        # best.pt 는 그대로 검증 점수로 고른다.
        patience=0,
        imgsz=640,
        seed=seed,
        deterministic=True,
        project=str(DRIVE / "detector" / "runs"),
        name=name,
    )
    train_minutes[name] = (time.time() - started) / 60
    weights_dir = DRIVE / "detector" / "runs" / name / "weights"
    sizes = [f"{f.name} {f.stat().st_size / 1e6:.1f}MB" for f in sorted(weights_dir.glob("*.pt"))]
    note(f"- {name}(묶음 {data} · 시드 {seed}): {train_minutes[name]:.1f}분 · " + ", ".join(sizes))

In [ ]:
# 5. best.pt 를 검증 라벨(사람이 검수한 d05)로 재고, 결과 zip · 보고서 마무리
note()
note("## 학습마다 best.pt (검증 d05 사람 라벨)")
note("| 학습 | 묶음 | 시드 | best epoch | P | R | mAP50 | mAP50-95 | 분 |")
note("|---|---|---|---|---|---|---|---|---|")
per_class = []
for name, (data, seed) in RUNS.items():
    run_dir = DRIVE / "detector" / "runs" / name
    with (run_dir / "results.csv").open(encoding="utf-8") as f:
        rows = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
    # best.pt 를 고르는 기준(fitness)이 mAP50-95 다(ultralytics/utils/metrics.py:1011)
    best = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
    best_epoch = int(float(best["epoch"]))
    metrics = YOLO(str(run_dir / "weights" / "best.pt")).val(
        data=str(WORK / data / f"yolo_{data}" / "data.yaml"),
        imgsz=640,
        plots=False,
        project=str(WORK / "val"),
        name=name,
    )
    box = metrics.box
    note(
        f"| {name} | {data} | {seed} | {best_epoch} | {box.mp:.3f} | {box.mr:.3f} | "
        f"{box.map50:.3f} | {box.map:.3f} | {train_minutes[name]:.1f} |"
    )
    classes = " · ".join(f"{row['Class']} {row['mAP50']:.3f}" for row in metrics.summary())
    per_class.append(f"- {name}: {classes}")
note()
note("종류별 mAP50:")
for line in per_class:
    note(line)
note()
note("원래 R2 · R3 는 T4 · CPU 8 에서 학습해 GPU 가 달라, 시드 0 도 그대로 견주지 않는다.")

# 맥으로 받을 파일 하나. 이름을 학습마다 달리 붙인다 — run_ablation 은 파일 이름을 표 이름으로
# 쓰고(scripts/run_ablation.py:310), Drive 에서 폴더 6개를 따로 받으면 놓치기 쉽다.
local_zip = WORK / OUT_ZIP.name
with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for name in RUNS:
        run_dir = DRIVE / "detector" / "runs" / name
        z.write(run_dir / "weights" / "best.pt", f"{name}_best.pt")
        z.write(run_dir / "weights" / "last.pt", f"{name}_last.pt")
        z.write(run_dir / "results.csv", f"{name}_results.csv")
shutil.copy2(local_zip, OUT_ZIP)  # zip 은 로컬에서 만들고 Drive 로는 한 번에 옮긴다
note()
note(f"결과 zip: {OUT_ZIP} ({OUT_ZIP.stat().st_size / 1e6:.0f}MB · 파일 {len(RUNS) * 3}개)")
note(
    "학습 우연의 폭 · 묶음 비교(후보 재현율 · 정답률)는 맥에서 run_ablation.py --weights 로 "
    "R0 · R2 · R3 와 같이 잰다."
)
print(f"\n끝. Drive JARVISEO/{REPORT.name} 와 {OUT_ZIP.name} 을 내려받는다.")

## 끝나면

- 맨 아래에 `끝.` 이 찍혔으면 성공이다.
  1. Drive `JARVISEO/SEEDS_report.txt` 와 `JARVISEO/JARVISEO_seeds_weights.zip` 을 내려받아 Claude 에게 알린다.
  2. 맥에서 zip 을 레포 `data/datasets/detector/runs/seeds/` 에 푼다 → `r2_s0_best.pt` ~ `r3_s2_last.pt` 12개와 `*_results.csv` 6개가 생긴다.
  3. 맥에서 R0 · R2 · R3 와 한 실행으로 잰다(`tools/README.md` "시드 반복").
- 중간에 빨간 오류로 멈췄으면 고치지 말고, 오류가 난 셀 번호와 마지막 20줄을 그대로 보낸다.